In [1]:
from pydantic import BaseModel, Field
from typing import List

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [2]:
# 1. Define Strict Schema
class MedicalTriage(BaseModel):
    patient_condition: str = Field(
        description="Summary of the patient's condition"
    )

    urgency_level: str = Field(
        description="Must be exactly 'LOW', 'MEDIUM', or 'HIGH'"
    )

    required_devices: List[str] = Field(
        description="List of medical devices or imaging tools needed (e.g., MRI, Dialysis Machine, ECG)"
    )

In [3]:
# 2. Setup Ollama LLM and Parser
llm = ChatOllama(
    model="mistral",
    temperature=0
)

parser = PydanticOutputParser(
    pydantic_object=MedicalTriage
)

In [4]:
# 3. Create Prompt
template = """
You are an AI diagnostic routing assistant.

Analyze the clinical text and return ONLY a JSON object.

Rules:
- Do not return JSON schema.
- Do not use keys like "properties", "title", or "type".
- urgency_level must be exactly LOW, MEDIUM, or HIGH.
- required_devices must be a JSON array of strings.

Clinical Text:
{clinical_text}

{format_instructions}
"""

In [5]:
prompt = PromptTemplate(
    template=template,
    input_variables=["clinical_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

In [6]:
# 4. LCEL Chain
triage_chain = prompt | llm | parser

In [7]:
# 5. Execute
patient_input = """
Patient arrived with severe lower back pain radiating to the abdomen.
Suspected renal calculi. Needs immediate scan to verify stone location.
"""

print("Processing Medical Data...\n")

result = triage_chain.invoke({
    "clinical_text": patient_input
})

Processing Medical Data...



In [8]:
# Output as a Pydantic Object
print("Parsed Object Type:", type(result))

print(f"Condition: {result.patient_condition}")
print(f"Urgency: {result.urgency_level}")
print(f"Devices to Ready: {', '.join(result.required_devices)}")

Parsed Object Type: <class '__main__.MedicalTriage'>
Condition: Severe lower back pain radiating to the abdomen. Suspected renal calculi. Needs immediate scan to verify stone location.
Urgency: HIGH
Devices to Ready: Scan device
